# Day 4: Containers with Docker

**Course:** Computational Workflows for Biomedical Data (CWBD), University of Tübingen, 2026 \
**Date:** 2026-10-01 \
**Author:** Laurens Hahn

**What:** Run, inspect and remove a first container, install FastQC by hand and then run it from a Seqera container, and build two images from Dockerfiles (cowsay, salmon 1.5.2).

**Why:** Every process of the nf-core runs on days 1 to 3 ran inside a container. Knowing what an image is and how one is built explains why those runs are reproducible, and where that stops.

## Docker basics

### Is Docker installed?

> Check if Docker is installed.

Yes, client and daemon both answer (version below; the full `docker info` also lists almost 40 images that the nf-core pipelines pulled on days 1 to 3).

In [41]:
!docker --version
!docker info 2>/dev/null | grep -E 'Server Version|Containers:|Images:'

Docker version 29.1.3, build 29.1.3-0ubuntu4.1
 Containers: 4
 Images: 51
 Server Version: 29.1.3


### What is a container?

> What is a container?

A container is an isolated process on the host that sees its own filesystem: the tool, its libraries and a minimal Linux userland, unpacked from an image. Unlike a virtual machine it does not boot its own operating system. It shares the host's kernel, and the kernel keeps it apart through namespaces (own process tree, mounts, network) and limits its CPU and memory through cgroups. That is why it starts in well under a second.

### Why do we use containers?

> Why do we use containers?

- **Reproducibility:** the image fixes the tool version and every library below it, so the same image gives the same result on my laptop, a cluster or the cloud
- **No installation and no conflicts:** each tool brings its own dependencies -> nf-core/rnaseq on day 2 ran about 20 tools, each in its own image, without installing any of them on my server
- **Shareable and versioned:** an image has a name, a tag and an immutable sha256 digest and sits in a registry, so a pipeline can point to exactly one build of a tool

### What is a docker image?

> What is a docker image?

A read only template for containers. It is a stack of filesystem layers, one per Dockerfile instruction that changes files, plus metadata such as environment variables and the default command. Starting a container adds a thin writable layer on top. Images are named `registry/repository:tag` (for example `quay.io/biocontainers/fastqc:0.12.1--hdfd78af_0`) and are stored in registries like Docker Hub, quay.io or community.wave.seqera.io.

## A first container

> Let's run our first docker image:

### Login to docker

> Login to docker

On the command line, `docker login -u <docker hub user>` asks for a personal access token and stores it in `~/.docker/config.json`. Pulling public images works without login; a login only raises Docker Hub's pull rate limit and is needed to push images. All images today are public.

In [42]:
# This you need to do on the command line directly
# docker login -u <docker hub user>

### Run your first docker container

> Run your first docker container

The image is not on the server yet, so Docker pulls it from Docker Hub, creates a container, runs its only program (`/hello`) and the container exits.

In [ ]:
!docker run hello-world

### Find the container ID

> Find the container ID

`docker ps` lists only running containers, and hello-world has already exited, so `-a` is needed.

In [44]:
!docker ps -a --filter ancestor=hello-world

CONTAINER ID   IMAGE         COMMAND    CREATED        STATUS                              PORTS     NAMES
f3a11036f239   hello-world   "/hello"   1 second ago   Exited (0) Less than a second ago             relaxed_meitner


### Delete the container and prove it is deleted

> Delete the container again, give prove its deleted

`docker rm` with the ID from above, then the same listing comes back empty. The image itself stays until `docker rmi`.

In [45]:
ids = !docker ps -aq --filter ancestor=hello-world
!docker rm {' '.join(ids)}

f3a11036f239


In [46]:
!docker ps -a --filter ancestor=hello-world
!docker rmi hello-world

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
Untagged: hello-world:latest
Deleted: sha256:5e23090353324d887c48ad5e5c56d294eab81588df9605b07d1afe895f9cc8f8


## FastQC by hand and in a container

### Running FastQC from the command line

> FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line \
> Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/ \
> Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. **Example file:** I downloaded read 1 of SRR23195511 (Sham_Oxy, one of the two runs from day 2) again from ENA: `wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR231/011/SRR23195511/SRR23195511_1.fastq.gz` into `data/` (1.8 GB, 21.5 M reads)
2. **FastQC:** downloaded `fastqc_v0.12.1.zip` from the Babraham page and unzipped it into `tools/FastQC/`. The zip holds the Java program plus a `fastqc` wrapper script
3. **Java:** FastQC needs a Java runtime, and my server has none outside the `cwbd` conda environment, which only provides Java while it is active. Instead of installing Java system wide, I unpacked the Eclipse Temurin JRE 21 tarball from adoptium.net into `tools/` next to FastQC. That keeps the install contained in the day 4 folder: nothing changes for other tools on the server, and deleting `tools/` removes it completely
4. **Run:** put that Java on `PATH` for the command only and ran the wrapper, output to `fastqc_manual/`

In [47]:
!mkdir -p fastqc_manual
!PATH="$PWD/tools/jdk-21.0.12.1+1-jre/bin:$PATH" tools/FastQC/fastqc --version
!PATH="$PWD/tools/jdk-21.0.12.1+1-jre/bin:$PATH" tools/FastQC/fastqc -q -o fastqc_manual data/SRR23195511_1.fastq.gz
!ls fastqc_manual

FastQC v0.12.1
application/gzip
SRR23195511_1_fastqc.html  SRR23195511_1_fastqc.zip


### FastQC from a Seqera container

> Very well, now let's try to make use of its docker container \
> 1. create a container holding fastqc using seqera containers (https://seqera.io/containers/) \
> 2. use the container to generate a fastqc html of the example fastq file

On seqera.io/containers: search `fastqc`, pick `bioconda::fastqc=0.12.1`, keep Docker and linux/amd64, "Get container". Seqera had this exact environment built already and returned `community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae`.

For the run, `-v "$PWD":/data` mounts the day 4 folder into the container (it cannot see host files otherwise), `-w /data` makes it the working directory, `-u` runs as me so the reports are not owned by root, and `--rm` removes the container afterwards.

In [48]:
# pull the container
!docker pull -q community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae


In [49]:
# run the container and save the results to a new "fastqc_results" directory
!mkdir -p fastqc_results
!docker run --rm -u $(id -u):$(id -g) -v "$PWD":/data -w /data community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae fastqc -q -o fastqc_results data/SRR23195511_1.fastq.gz
!ls fastqc_results

application/gzip
SRR23195511_1_fastqc.html  SRR23195511_1_fastqc.zip


### Which approach was easier, now and in the future?

> Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

The container, already today: one pull and one run. By hand I had to find out that Java was missing, fetch a runtime and wire up `PATH`. In the future the gap grows: every further tool or version is just another image tag, while manual installs pile up on the host and can clash. The one thing the container needs is Docker itself.

### Which approach is more reproducible?

> What would you say, which approach is more reproducible?

The container. Its tag pins FastQC, the Java runtime and the OS libraries together, and the digest makes it immutable. My manual install depends on whatever "latest JRE 21" was today and on the host. One caveat from the Dockerfile part below: a Dockerfile that runs `apt-get install` against live repositories is not reproducible by itself; the built and pushed image is.

### Are the reports identical to the earlier FastQC results?

> Compare the file to last weeks fastqc results, are they identical? \
> Is the fastqc version identical?

The earlier result is the raw read FastQC that nf-core/rnaseq ran on day 2 (`quay.io/biocontainers/fastqc:0.12.1`), on the same file renamed to `Sham_Oxy_SRR23195511_raw_1.fastq.gz`. I compare `fastqc_data.txt` from the three zips line by line.

In [50]:
import zipfile

reports = {
    "manual": "fastqc_manual/SRR23195511_1_fastqc.zip",
    "seqera container": "fastqc_results/SRR23195511_1_fastqc.zip",
    "day 2 nf-core/rnaseq": "../day_02/results/rnaseq/fastqc/raw/Sham_Oxy_SRR23195511_raw_1_fastqc.zip",
}

def fastqc_data(path):
    with zipfile.ZipFile(path) as z:
        name = next(n for n in z.namelist() if n.endswith("fastqc_data.txt"))
        return z.read(name).decode().splitlines()

data = {k: fastqc_data(p) for k, p in reports.items()}
ref = data["manual"]
for k, lines in data.items():
    diff = [(a, b) for a, b in zip(ref, lines) if a != b]
    print(f"{k}: {lines[0]}, {len(lines)} lines, differing lines vs manual: {diff}")

manual: ##FastQC	0.12.1, 4025 lines, differing lines vs manual: []
seqera container: ##FastQC	0.12.1, 4025 lines, differing lines vs manual: []
day 2 nf-core/rnaseq: ##FastQC	0.12.1, 4025 lines, differing lines vs manual: [('Filename\tSRR23195511_1.fastq.gz', 'Filename\tSham_Oxy_SRR23195511_raw_1.gz')]


Yes, apart from the file name. All three reports come from FastQC 0.12.1, and all 4025 lines of `fastqc_data.txt` match between the manual run and the Seqera container. The day 2 report differs in exactly one line, `Filename`, because nf-core/rnaseq renames the input to `Sham_Oxy_SRR23195511_raw_1`. So the manual install with its own Java 21, the Seqera image and the biocontainers image in the pipeline give the same numbers. The HTML files are not byte identical, since they carry the file name and the date the report was made.

## Dockerfiles

### A toy image with cowsay

> We now used Docker containers and images directly to boost our research. \
> Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay) \
> Hints: \
> 1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor
!cat my_dockerfile

### The RUN and ENV lines

> Explain the RUN and ENV lines you added to the file

- **RUN** executes a shell command at build time inside the image being built; its file changes become a new layer. `apt-get update` fetches the package lists (the slim image ships without them), `apt-get install -y` installs curl and cowsay without asking, `--no-install-recommends` skips optional extras, and deleting `/var/lib/apt/lists` keeps the layer small. All three are chained with `&&` in one RUN, so a cached old `update` layer can never be combined with a newer `install`.
- **ENV** sets an environment variable in the image, valid in every container started from it. Debian installs cowsay to `/usr/games`, which is not on the image's default `PATH`, so `cowsay` alone gives "not found". I prepend `/usr/games` instead of setting `PATH=/usr/games` alone, which would hide `/bin` and `/usr/bin` and with them every normal command.
- **FROM:** I changed the provided `debian:bullseye-slim` to `debian:trixie-slim` (Debian 13, the current stable release). Bullseye reached end of life in August 2026; `apt-get update` still works, but the security repository answers 404 for the curl packages it lists, so the build fails.
- A `.dockerignore` keeps `data/` (the 1.8 GB FASTQ) and `tools/` out of the build context, which Docker otherwise sends to the daemon in full.

In [52]:
# build the docker image
!docker build -q -t cowsay -f my_dockerfile .

sha256:141c1e045e740d09cecec29674a79d759cc64842e20f515759575efd91bdb346


In [53]:
# make sure that the image has been built
!docker images cowsay

                                                            i Info →   U  In Use
IMAGE           ID             DISK USAGE   CONTENT SIZE   EXTRA
cowsay:latest   141c1e045e74        205MB           47MB        


In [54]:
# run the docker file 
!docker run --rm cowsay cowsay "Hello, World!"

 _______________
< Hello, World! >
 ---------------
        \   ^__^
         \  (oo)\_______
            (__)\       )\/\
                ||----w |
                ||     ||


## Salmon in a self built image

> Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq \
> To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

Same base as cowsay, plus `ca-certificates` so curl trusts GitHub's certificate. The release tarball is streamed straight into `tar`, so it never lands in a layer, and the `bin` folder goes on `PATH`. The tarball also carries salmon's own `lib/` folder, so no further packages are needed.

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image
!cat salmon_docker

In [56]:
# build the image
!docker build -q -t salmon:1.5.2 -f salmon_docker .

sha256:8a5ddfb509659e87b6c71676509dca76add435334fdfeedeb544f32097416317


In [57]:
# run the docker image to give out the version of salmon
!docker run --rm salmon:1.5.2 salmon --version

salmon 1.5.2


## Do we always need our own image?

> Do you think bioinformaticians have to create a docker image every time they want to run a tool? \
> Find the salmon docker image online and run it on your computer. \
> What is https://biocontainers.pro/ ?

No. For almost every tool on Bioconda an image already exists. BioContainers is the community project behind them: it builds a Docker image for every Bioconda package automatically (on quay.io/biocontainers, Singularity images on depot.galaxyproject.org), tagged `<version>--<build>`. The salmon 1.5.2 image is `quay.io/biocontainers/salmon:1.5.2--h84f40af_0`; nf-core modules used these images for years.

In [58]:
!docker pull -q quay.io/biocontainers/salmon:1.5.2--h84f40af_0
!docker run --rm quay.io/biocontainers/salmon:1.5.2--h84f40af_0 salmon --version
!docker images 2>/dev/null | grep -E 'salmon|REPOSITORY'

quay.io/biocontainers/salmon:1.5.2--h84f40af_0
salmon 1.5.2
community.wave.seqera.io/library/salmon:2.7.0--74784226202c61b9                                       8b30cea85887        211MB         49.2MB        
quay.io/biocontainers/salmon:1.5.2--h84f40af_0                                                        4ae09a47788f        372MB         62.4MB        
salmon:1.5.2                                                                                          8a5ddfb50965        409MB         88.5MB        


## Other ways to build images

> Are there other ways to create Docker (or Apptainer) images? \
> What is https://seqera.io/containers/ ?

Seqera Containers builds images on demand from a list of conda (Bioconda, conda-forge) or PyPI packages, several tools in one image if needed, as Docker or Singularity image. The builds run on Seqera's Wave service and land in the free public registry community.wave.seqera.io. The tag ends in a hash of the environment, so the same request returns the same, already built image; that happened with FastQC above. New nf-core modules point to these images: most of what days 2 and 3 pulled comes from community.wave.seqera.io.